In [ ]:
import pandas as pd
import numpy as np

# 1. Función inteligente de limpieza socio-demográfica
def limpiar_socio(path, pivot_col):
    df = pd.read_csv(path)
    
    # Arreglo especial para el archivo de Población Inmigrante ("01 ene YYYY")
    if 'Poblacion' in path:
        nuevas_cols = []
        for col in df.columns:
            if '01 ene' in str(col).lower():
                nuevas_cols.append(str(col).split()[-1]) # Extrae solo el año
            else:
                nuevas_cols.append(col)
        df.columns = nuevas_cols
        
        # Filtramos solo Municipios y Distritos para evitar duplicar población con los "barrios"
        df = df[df['Tipo de territorio'].isin(['Municipi', 'Districte'])]
    
    # Identificar columnas que representan años (excluyendo texto base)
    columnas_años = [c for c in df.columns if c not in ['Territorio', 'Tipo de territorio', pivot_col]]
    
    # Limpiar caracteres como comas o guiones y forzar formato número decimal
    for col in columnas_años:
        df[col] = df[col].astype(str).str.replace(',', '.', regex=False)\
                                     .str.replace('-', '0', regex=False)\
                                     .str.replace(' ', '', regex=False)\
                                     .replace('', '0')\
                                     .astype(float)
    
    # Convertir formato matriz a formato de lista (Melt -> Pivot)
    melted = df.melt(id_vars=['Territorio', 'Tipo de territorio', pivot_col], var_name='Año', value_name='Val')
    melted['Año'] = pd.to_numeric(melted['Año'], errors='coerce')
    
    # Re-pivotar organizando por columnas limpias
    pivotado = melted.pivot_table(index=['Territorio', 'Año'], columns=pivot_col, values='Val', aggfunc='first').reset_index()
    return pivotado

print("🧹 Limpiando y homogeneizando datasets sociales...")
df_edad = limpiar_socio('Edad_Barrios.csv', 'Edad en grandes grupos')
df_pob = limpiar_socio('Poblacion-Inmigrante.csv', 'Nacionalidad (España/UE/Resto extranjero)')

# 2. Unión Maestra con el histórico de criminalidad
df_crime = pd.read_csv('Criminalidad_Mensual_Estructurada.csv')
df_m = pd.merge(df_crime, df_edad, on=['Territorio', 'Año'], how='left')
df_m = pd.merge(df_m, df_pob, on=['Territorio', 'Año'], how='left')

# 3. Reparación de Datos del 2026 y Funciones de Ciclo Temporal
social_cols = ['16-64 años', '<16 años', '≥65 años', 'España', 'Resto de la Unión Europea', 'Resto del mundo']
df_m = df_m.sort_values(['Territorio', 'Categoría_Delito', 'Año', 'Mes_Num'])

# "Forward Fill" (Copiar los datos de 2025 al 2026 asumiendo estabilidad demográfica anual)
df_m[social_cols] = df_m.groupby(['Territorio', 'Categoría_Delito'])[social_cols].ffill()

# Si aún tras la reparación queda algún NaN (ej. Distritos sin informar), se elimina
df_m = df_m.dropna(subset=social_cols)

# Codificación Cíclica: Ayuda a la IA a entender que Diciembre y Enero están conectados temporalmente
df_m['mes_sin'] = np.sin(2 * np.pi * df_m['Mes_Num'] / 12)
df_m['mes_cos'] = np.cos(2 * np.pi * df_m['Mes_Num'] / 12)

# Exportación Final
df_m.to_csv('Criminalidad_Elite_Limpia.csv', index=False)

print(f"✅ ¡Dataset Elite creado con éxito! Se han combinado {len(df_m)} registros sin errores.")

In [ ]:
from sklearn.preprocessing import StandardScaler

class EliteDataEngine:
    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        self.features = ['Cantidad', 'mes_sin', 'mes_cos', '16-64 años', 'España', 'Resto del mundo']
        self.scaler = StandardScaler()

    def prepare(self, distrito, delito, window=12):
        data = self.df[(self.df['Territorio'] == distrito) & 
                       (self.df['Categoría_Delito'] == delito)].sort_values(['Año', 'Mes_Num'])
        
        if len(data) < window + 2: return None, None, None
        
        raw_values = data[self.features].values
        scaled_data = self.scaler.fit_transform(raw_values)
        
        X, y = [], []
        for i in range(len(scaled_data) - window):
            X.append(scaled_data[i : i + window, :])
            y.append(scaled_data[i + window, 0])
            
        return np.array(X), np.array(y), self.scaler

engine = EliteDataEngine('Criminalidad_Elite_Limpia.csv')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam

def build_elite_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        GRU(128, return_sequences=True, kernel_regularizer='l2'),
        BatchNormalization(),
        Dropout(0.3),
        GRU(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    # Huber loss es excelente para evitar que errores puntuales arruinen el modelo
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='huber')
    return model

In [ ]:
from sklearn.metrics import r2_score

D_SELECT, C_SELECT = 'Eixample', 'Hurto'
X, y, scaler = engine.prepare(D_SELECT, C_SELECT)

split = int(len(X) * 0.85)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

model = build_elite_model((X.shape[1], X.shape[2]))
print(f"🚀 Entrenando modelo de alta precisión para {C_SELECT}...")

model.fit(X_train, y_train, epochs=120, batch_size=16, 
          validation_data=(X_test, y_test), verbose=0)

# Evaluación
preds = model.predict(X_test, verbose=0)
# Desescalar
dummy = np.zeros((len(preds), len(engine.features)))
dummy[:, 0] = preds.flatten()
inv_preds = scaler.inverse_transform(dummy)[:, 0]

dummy_r = np.zeros((len(y_test), len(engine.features)))
dummy_r[:, 0] = y_test
inv_real = scaler.inverse_transform(dummy_r)[:, 0]

print(f"📊 NUEVA PRECISIÓN (R²): {r2_score(inv_real, inv_preds):.4f}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Proyectar 2026
# (Aquí integrarías el bucle recursivo del paso anterior usando 'model' y 'scaler')
# Visualización
plt.figure(figsize=(12, 6))
plt.plot(inv_real, label='Real', marker='o', color='blue')
plt.plot(inv_preds, label='IA Optimizada', ls='--', color='red')
plt.title(f"Mejora del Modelo: {C_SELECT} en {D_SELECT}")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import r2_score

# Predecir sobre el set de test
y_pred_esc = model_final.predict(X_test)

# Desescalar solo la columna 0 (delitos)
dummy_p = np.zeros((len(y_pred_esc), 6))
dummy_p[:, 0] = y_pred_esc.flatten()
inv_pred = sc.inverse_transform(dummy_p)[:, 0]

dummy_r = np.zeros((len(y_test), 6))
dummy_r[:, 0] = y_test
inv_real = sc.inverse_transform(dummy_r)[:, 0]

# Cálculo de precisión real
r2 = r2_score(inv_real, inv_pred)
mape = np.mean(np.abs((inv_real - inv_pred) / inv_real)) * 100

print(f"\n📈 RESULTADOS DE MEJORA:")
print(f"Precisión (R² Score): {r2:.4f}")
print(f"Error Medio Porcentual: {mape:.2f}%")

if r2 > 0.80:
    print("💎 ¡Excelente! La red ha captado los patrones sociales perfectamente.")
else:
    print("⚠️ El modelo ha mejorado, pero la criminalidad tiene componentes aleatorios altos.")